# 02 Financial Health Scoring
## Nifty Financial Platform

This notebook explores the logic used to calculate financial health scores for companies. It includes feature engineering, normalization, and the final scoring algorithm.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from sklearn.preprocessing import MinMaxScaler
from dotenv import load_dotenv

# Settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load environment variables
load_dotenv(dotenv_path='../.env')
DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    print("ERROR: DATABASE_URL not found in .env file")
else:
    engine = create_engine(DATABASE_URL)
    print("Database engine created successfully.")

## 1. Fetch Raw Features
We pull raw metrics from Profit & Loss, Balance Sheet, and Cash Flow tables.

In [ ]:
def fetch_scoring_data(engine):
    query = """
    SELECT 
        dc.symbol, dc.company_name, pl.company_id, pl.year_id,
        pl.revenue, pl.net_profit,
        bs.total_assets, bs.borrowings, bs.equity_capital, bs.reserves,
        cf.operating_cash_flow
    FROM fact_profit_loss pl
    JOIN fact_balance_sheet bs ON pl.company_id = bs.company_id AND pl.year_id = bs.year_id
    JOIN fact_cash_flow cf ON pl.company_id = cf.company_id AND pl.year_id = cf.year_id
    JOIN dim_company dc ON pl.company_id = dc.company_id
    """
    return pd.read_sql(query, engine)

df = fetch_scoring_data(engine)
df = df.sort_values(by=['symbol', 'year_id']).reset_index(drop=True)
df.head()

## 2. Feature Engineering
Calculated ratios that represent financial health.

In [ ]:
def safe_div(n, d):
    return np.where(d == 0, 0.0, n / d)

df['net_profit_margin_pct'] = safe_div(df['net_profit'], df['revenue'])
df['return_on_assets'] = safe_div(df['net_profit'], df['total_assets'])
df['debt_to_equity'] = safe_div(df['borrowings'], df['equity_capital'] + df['reserves'])
df['equity_ratio'] = safe_div(df['equity_capital'] + df['reserves'], df['total_assets'])
df['cash_conversion_ratio'] = safe_div(df['operating_cash_flow'], df['net_profit'])
df['revenue_growth'] = df.groupby('symbol')['revenue'].pct_change().fillna(0)

features = ['net_profit_margin_pct', 'return_on_assets', 'revenue_growth', 'debt_to_equity', 'equity_ratio', 'cash_conversion_ratio']
df[features].describe()

## 3. Preprocessing (Outliers & Scaling)
We clip outliers and normalize features to a 0-1 range.

In [ ]:
# Clip outliers at 1st and 99th percentiles
for col in features:
    df[col] = df[col].clip(lower=df[col].quantile(0.01), upper=df[col].quantile(0.99))

# Normalize
scaler = MinMaxScaler()
df_norm = pd.DataFrame(scaler.fit_transform(df[features]), columns=features)

# Invert Debt to Equity (Lower is better)
df_norm['debt_to_equity'] = 1.0 - df_norm['debt_to_equity']

df_norm.head()

## 4. Scoring Logic
Weighted average of different financial pillars.

In [ ]:
prof_score = (df_norm['net_profit_margin_pct'] + df_norm['return_on_assets']) / 2
growth_score = df_norm['revenue_growth']
lev_score = (df_norm['debt_to_equity'] + df_norm['equity_ratio']) / 2
cf_score = df_norm['cash_conversion_ratio']

df['overall_score'] = (prof_score * 0.40) + (growth_score * 0.25) + (lev_score * 0.15) + (cf_score * 0.20)

# Classify into labels based on quartiles
q1, q2, q3 = df['overall_score'].quantile([0.25, 0.5, 0.75])

def classify(s):
    if s >= q3: return 'EXCELLENT'
    if s >= q2: return 'GOOD'
    if s >= q1: return 'AVERAGE'
    return 'WEAK'

df['health_label'] = df['overall_score'].apply(classify)
sns.countplot(data=df, x='health_label', order=['EXCELLENT', 'GOOD', 'AVERAGE', 'WEAK'], palette='RdYlGn_r')
plt.title('Distribution of Financial Health Labels')
plt.show()

## 5. Visualizing Scores for Top Companies

In [ ]:
top_companies = df.sort_values('overall_score', ascending=False).head(15)
sns.barplot(data=top_companies, x='overall_score', y='company_name', hue='health_label', dodge=False)
plt.title('Top 15 Companies by Financial Health Score')
plt.legend(title='Health Label', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

## 6. Export Results to Database
We can update the `fact_ml_scores` table with these new calculations.

In [ ]:
from sqlalchemy import text

def export_scores(df, engine):
    with engine.begin() as conn:
        # Ensure labels exist
        labels = df['health_label'].unique()
        for lbl in labels:
            conn.execute(text("INSERT INTO dim_health_label (label_name) VALUES (:l) ON CONFLICT DO NOTHING"), {"l": lbl})
        
        label_map = dict(conn.execute(text("SELECT label_name, health_id FROM dim_health_label")).fetchall())
        df['health_id'] = df['health_label'].map(label_map)
        
        for _, row in df.iterrows():
            conn.execute(text("""
                INSERT INTO fact_ml_scores (company_id, year_id, health_id, probability_score)
                VALUES (:cid, :yid, :hid, :score)
                ON CONFLICT (company_id, year_id) DO UPDATE SET
                    health_id = EXCLUDED.health_id,
                    probability_score = EXCLUDED.probability_score,
                    prediction_date = CURRENT_TIMESTAMP
            """), {
                "cid": int(row['company_id']), 
                "yid": int(row['year_id']), 
                "hid": int(row['health_id']), 
                "score": float(row['overall_score'])
            })
    print("Scores exported to fact_ml_scores successfully.")

# Uncomment to export
# export_scores(df, engine)